In [129]:
import pandas as pd 
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import make_scorer, accuracy_score, f1_score, confusion_matrix
from sklearn.ensemble import GradientBoostingClassifier 

In [130]:
data = pd.read_csv('../dataset/final_dataset_diff_f_L10.csv', parse_dates=['GAME_DATE'], dtype={'gameId' : str, 'H_teamId' : str, 'A_teamId' : str,})
data = data.round(2)

In [131]:
condition = (data['GAME_DATE'] > pd.to_datetime('2023-09-01')) & (data['GAME_DATE'] < pd.to_datetime('2024-09-01'))
data_test = data[condition]
#data_test = data_test.drop(columns=['GAME_DATE','gameId', 'A_teamId', 'H_teamId'])

data_train = data[~condition]
#data_train = data_train.drop(columns=['GAME_DATE','gameId', 'A_teamId', 'H_teamId'])

In [132]:
X_train = data_train.drop(columns=['HOME_WON', 'GAME_DATE','gameId', 'A_teamId', 'H_teamId','H_teamId','A_teamId','H_POINTS', 'A_POINTS','H_teamName', 'A_teamName','trueShootingPercentage_L10', 'PCT_TIR_REUSSI_L10', 'effectiveFieldGoalPercentage_L10','NB_WIN_L10', 'PCT_3PT_L10', 'W_CONFR_D'])  # Fonctionnalités
y_train = data_train['HOME_WON']  # Cible

X_test = data_test.drop(columns=['HOME_WON', 'GAME_DATE','gameId', 'A_teamId', 'H_teamId','H_teamId','A_teamId','H_POINTS', 'A_POINTS','H_teamName', 'A_teamName','trueShootingPercentage_L10', 'PCT_TIR_REUSSI_L10', 'effectiveFieldGoalPercentage_L10', 'NB_WIN_L10', 'PCT_3PT_L10', 'W_CONFR_D'])  # Fonctionnalités
y_test = data_test['HOME_WON']

scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.fit_transform(X_test)

In [133]:
# Définir les métriques de performance à calculer
scoring = {'accuracy': make_scorer(accuracy_score), 'f1': make_scorer(f1_score)}

In [134]:
gb_regressor = GradientBoostingClassifier()


In [135]:
gb_regressor.fit(X_train, y_train)
y_pred = gb_regressor.predict(X_test)

In [136]:
accuracy = accuracy_score(y_test, y_pred)  # y_test sont les étiquettes de classe réelles des données de test
f1 = f1_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)

print("Accuracy:", accuracy)
print("F1-score:", f1)
print("Matrice de confusion :")
print(conf_matrix)

Accuracy: 0.6470588235294118
F1-score: 0.7136563876651982
Matrice de confusion :
[[229 274]
 [116 486]]


In [137]:
data_knn_predict = data_test[['gameId', 'GAME_DATE', 'HOME_WON', 'H_POINTS', 'A_POINTS','H_teamName', 'A_teamName']].copy()
data_knn_predict.loc[:, 'PRED'] = y_pred
#data_knn_predict = data_knn_predict.iloc[[2,3, 50,51, 207,208]]
data_knn_predict = data_knn_predict[['gameId', 'GAME_DATE', 'HOME_WON', 'PRED', 'H_POINTS', 'A_POINTS','H_teamName', 'A_teamName']]
data_knn_predict

,gameId,GAME_DATE,HOME_WON,PRED,H_POINTS,A_POINTS,H_teamName,A_teamName
10794,0022300001,2023-11-03,1,1,121,116,Indiana Pacers,Cleveland Cavaliers
10795,0022300002,2023-11-03,1,1,110,105,Milwaukee Bucks,New York Knicks
10796,0022300003,2023-11-03,1,1,121,114,Miami Heat,Washington Wizards
10797,0022300004,2023-11-03,0,1,107,109,Chicago Bulls,Brooklyn Nets
10798,0022300005,2023-11-03,0,1,139,141,Oklahoma City Thunder,Golden State Warriors
...,...,...,...,...,...,...,...,...
11894,0022301226,2023-12-08,0,1,112,125,Portland Trail Blazers,Dallas Mavericks
11895,0022301227,2023-12-08,1,1,133,123,Boston Celtics,New York Knicks
11896,0022301228,2023-12-08,0,1,106,114,Phoenix Suns,Sacramento Kings
11897,0022301229,2023-12-07,0,1,119,128,Milwaukee Bucks,Indiana Pacers


In [138]:
data_knn_predict.set_index('gameId', inplace=True)
data_knn_predict = data_knn_predict.sort_values(by='GAME_DATE')
data_knn_predict.transpose().to_json("../dataset/KNN_predict.json")